# CardioScore Validation 00 — Intake & Provenance

Fail-closed intake for public validation sources. Do not edit the downloaded source.

In [ ]:

import sys, subprocess, json, hashlib, zipfile, tarfile
import pandas as pd
from pathlib import Path
PIN = "869150cd5fb5ccf155fb066258404bd4df163ade"
REPO = "Virelion-Biotech/Virelion-CardioScore"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"git+https://github.com/{REPO}.git@{PIN}"], check=True)
print("Installed pinned CardioScore:", PIN)


In [ ]:

from google.colab import files
up = files.upload()
RAW = Path("/content/cardioscore_validation/raw")
RAW.mkdir(parents=True, exist_ok=True)
for name, data in up.items():
    (RAW/name).write_bytes(data)
print("Uploaded:", [p.name for p in RAW.iterdir()])


In [ ]:

def sha256(path, chunk=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

def inventory(path):
    path = Path(path)
    if zipfile.is_zipfile(path):
        with zipfile.ZipFile(path) as z:
            return [{"name": i.filename, "bytes": i.file_size, "crc": i.CRC} for i in z.infolist()]
    if tarfile.is_tarfile(path):
        with tarfile.open(path, "r:*") as t:
            return [{"name": i.name, "bytes": i.size, "type": i.type.decode(errors="replace")} for i in t.getmembers()]
    return [{"name": path.name, "bytes": path.stat().st_size}]

sources = list(RAW.iterdir())
assert len(sources) == 1, "Upload exactly one authoritative source asset in this intake cell."
src = sources[0]
receipt = {
    "source_id": "REPLACE_ME",
    "source_url": "REPLACE_ME",
    "source_filename": src.name,
    "source_path": str(src),
    "byte_size": src.stat().st_size,
    "sha256": sha256(src),
    "inventory": inventory(src)
}
DERIVED = Path("/content/cardioscore_validation/derived")
DERIVED.mkdir(parents=True, exist_ok=True)
(DERIVED/"source_receipt.json").write_text(json.dumps(receipt, indent=2)+"\n")
print(json.dumps(receipt, indent=2))


In [ ]:

import pandas as pd
src = sources[0]
suf = src.suffix.lower()
if suf in {".xlsx", ".xls"}:
    x = pd.ExcelFile(src)
    print("Workbook sheets:", x.sheet_names)
    for s in x.sheet_names:
        d = pd.read_excel(src, sheet_name=s, nrows=5)
        print("\n---", s, d.shape, "---")
        print(list(d.columns))
        print(d.head().to_string(index=False))
elif suf == ".csv":
    d = pd.read_csv(src, nrows=5)
    print("CSV columns:", list(d.columns))
    print(d.to_string(index=False))
elif suf in {".h5", ".hdf5"}:
    import h5py
    with h5py.File(src, "r") as h:
        def walk(g, prefix=""):
            for k, v in g.items():
                p = prefix + "/" + k
                print(p, type(v).__name__, getattr(v, "shape", None), getattr(v, "dtype", None))
                if hasattr(v, "items"):
                    walk(v, p)
        walk(h)
elif suf == ".mat":
    import scipy.io as sio
    m = sio.whosmat(src)
    print("MAT variables:", m)
else:
    print("No parser is applied for this source format. Inspect it manually and create a source-specific adapter.")


### Gate 00
Keep `source_receipt.json`. A later notebook may proceed only after the exact source file, SHA-256, and source-specific adapter are known.